In [ ]:
import json, re, hashlib, random
from dataclasses import dataclass
from typing import List, Optional, Dict, Set

TOKEN_SPEC = [
    ("SPACE",   r"\s+"),
    ("ARROW2",  r"<->"),
    ("ARROW",   r"->"),
    ("NOT",     r"[!~¬]"),
    ("AND",     r"[&∧]"),
    ("OR",      r"[|∨]"),
    ("LPAREN",  r"\("),
    ("RPAREN",  r"\)"),
    ("COMMA",   r","),
    ("IDENT",   r"[A-Za-z_][A-Za-z0-9_]*"),
]
TOKEN_RE = re.compile("|".join(f"(?P<{name}>{pat})" for name, pat in TOKEN_SPEC))

@dataclass
class Token:
    kind: str
    text: str

def lex(s: str) -> List[Token]:
    pos, out = 0, []
    while pos < len(s):
        m = TOKEN_RE.match(s, pos)
        if not m:
            raise ValueError(f"Unexpected character at {pos}: {s[pos:pos+20]}")
        kind = m.lastgroup
        text = m.group(kind)
        pos = m.end()
        if kind == "SPACE":
            continue
        if kind == "IDENT" and text.lower() in ("implies", "equivalent"):
            text = text.capitalize()  # 规范成首字母大写
        out.append(Token(kind, text))
    return out

@dataclass
class Node:
    op: str
    a: Optional['Node'] = None
    b: Optional['Node'] = None
    name: Optional[str] = None
    kids: Optional[List['Node']] = None 


class Parser:
    def __init__(self, tokens: List[Token]):
        self.toks = tokens
        self.i = 0

    def peek(self) -> Optional[Token]:
        return self.toks[self.i] if self.i < len(self.toks) else None

    def eat(self, kind=None) -> Token:
        tok = self.peek()
        if tok is None:
            raise ValueError("Unexpected end of input")
        if kind and tok.kind != kind:
            raise ValueError(f"Expected {kind}, got {tok.kind} ({tok.text})")
        self.i += 1
        return tok

    def parse(self) -> Node:
        node = self.parse_equiv()
        if self.peek() is not None:
            raise ValueError(f"Trailing tokens: {self.toks[self.i:]}")
        return node

    def parse_equiv(self) -> Node:
        left = self.parse_implies()
        while True:
            tok = self.peek()
            if tok and tok.kind == "ARROW2":
                self.eat("ARROW2")
                right = self.parse_implies()
                left = Node("IFF", a=left, b=right)
            else:
                break
        return left

    def parse_implies(self) -> Node:
        left = self.parse_or()
        while True:
            tok = self.peek()
            if tok and tok.kind == "ARROW":
                self.eat("ARROW")
                right = self.parse_implies()  # 右结合
                left = Node("IMPLIES", a=left, b=right)
            else:
                break
        return left

    def parse_or(self) -> Node:
        items = [self.parse_and()]
        while True:
            tok = self.peek()
            if tok and tok.kind == "OR":
                self.eat("OR")
                items.append(self.parse_and())
            else:
                break
        return items[0] if len(items) == 1 else Node("OR", kids=items)

    def parse_and(self) -> Node:
        items = [self.parse_unary()]
        while True:
            tok = self.peek()
            if tok and tok.kind == "AND":
                self.eat("AND")
                items.append(self.parse_unary())
            else:
                break
        return items[0] if len(items) == 1 else Node("AND", kids=items)

    def parse_unary(self) -> Node:
        tok = self.peek()
        if tok and tok.kind == "NOT":
            self.eat("NOT")
            return Node("NOT", a=self.parse_unary())
        return self.parse_primary()

    def parse_primary(self) -> Node:
        tok = self.peek()
        if tok is None:
            raise ValueError("Unexpected end")
        if tok.kind == "LPAREN":
            self.eat("LPAREN")
            inside = self.parse_equiv()
            self.eat("RPAREN")
            return inside
        if tok.kind == "IDENT":
            ident = self.eat("IDENT").text
            if ident in ("Implies", "Equivalent") and self.peek() and self.peek().kind == "LPAREN":
                self.eat("LPAREN")
                a = self.parse_equiv()
                self.eat("COMMA")
                b = self.parse_equiv()
                self.eat("RPAREN")
                return Node("IMPLIES" if ident == "Implies" else "IFF", a=a, b=b)
            return Node("ATOM", name=ident)
        raise ValueError(f"Unexpected token: {tok.kind} {tok.text}")

def collect_atoms(n: Node, bag: Optional[Set[str]] = None) -> Set[str]:
    if bag is None:
        bag = set()
    if n.op == "ATOM":
        bag.add(n.name)
    else:
        if n.a is not None: collect_atoms(n.a, bag)
        if n.b is not None: collect_atoms(n.b, bag)
        if n.kids:
            for k in n.kids:
                collect_atoms(k, bag)
    return bag

def to_3sg(v: str) -> str:
    v = v.strip().lower()
    if not v:
        return v
    if v.endswith(("s", "x", "z", "ch", "sh", "o")):
        return v + "es"           # watch -> watches, go -> goes
    if v.endswith("y") and len(v) >= 2 and v[-2] not in "aeiou":
        return v[:-1] + "ies"     # try -> tries
    return v + "s"                # play -> plays


def negate_sentence(s: str) -> str:

    t = s.strip().rstrip(".")
    low = t.lower()

    # 1) The NOUN is ADJ
    # The sensor is active -> The sensor is not active
    if low.startswith("the ") and " is " in low:
        if " is not " in low:
            return t + "."
        return (t.replace(" is ", " is not ", 1)) + "."

    # 2) The NOUN occurs/fires/triggers/activates ...
    mono_verbs = ("occurs", "fires", "triggers", "activates", "appears", "stops", "rotates",
                  "vibrates", "drifts", "oscillates", "fails", "signals")
    if low.startswith("the "):
        parts = t.split()
        if len(parts) >= 2:
            v = parts[-1]
            vlow = v.lower()
            if vlow in mono_verbs or vlow.endswith("s"):
                if vlow.endswith("ies") and len(v) > 3:
                    base = v[:-3] + "y"
                elif vlow.endswith("es") and not vlow.endswith("ses"):  # 比如 rotates -> rotate, but 'passes' -> pass(es)…简化处理
                    base = v[:-2]
                elif vlow.endswith("s"):
                    base = v[:-1]
                else:
                    base = v
                parts[-1] = "does not " + base
                return " ".join(parts) + "."

    # 3)have/has：The X has … -> The X does not have …
    if low.startswith("the ") and (" has " in low or " have " in low):
        parts = t.split()
        for i, w in enumerate(parts):
            if w.lower() in ("has", "have"):
                parts[i] = "does not have"
                return " ".join(parts) + "."
    # 4) 
    return "It is not the case that the following holds: " + t + "."

def _roman(n: int) -> str:
    # 1->i, 2->ii, 3->iii, 4->iv, 5->v ...
    vals = [
        (1000, "m"), (900, "cm"), (500, "d"), (400, "cd"),
        (100, "c"), (90, "xc"), (50, "l"), (40, "xl"),
        (10, "x"), (9, "ix"), (5, "v"), (4, "iv"), (1, "i")
    ]
    out = []
    for v, s in vals:
        while n >= v:
            out.append(s)
            n -= v
    return "".join(out)

def render_en_bracketed(node: Node, atom_mapper: Optional[Dict[str, str]] = None) -> str:
    """
       AND -> First, … . and second, … . and third, … .
       OR  -> either (i) X OR (ii) Y OR (iii) Z
       NOT -> It is not the case that the following holds: 
    """
    ordinal_words = ["first", "second", "third", "fourth", "fifth", "sixth", "seventh"]

    def strip_period(s: str) -> str:
        return s.strip().rstrip(".")

    def R(n: Node) -> str:

        if n.op == "ATOM":
            return strip_period(atom_mapper.get(n.name, n.name) if atom_mapper else n.name)


        if n.op == "NOT":
            # ~~φ -> φ
            if n.a and n.a.op == "NOT":
                return R(n.a.a)
            if n.a and n.a.op == "ATOM":
                surface = atom_mapper.get(n.a.name, n.a.name) if atom_mapper else n.a.name
                return strip_period(negate_sentence(surface))
            inner = n.a
            if inner and inner.op == "OR":
                kids = inner.kids or []
                parts = [strip_period(R(k)) for k in kids]
                if len(parts) == 0:
                    tail = ""
                elif len(parts) == 1:
                    tail = parts[0]
                else:
                    items = [f"({ _roman(i+1) }) {p}" for i, p in enumerate(parts)]
                    tail = "either " + " OR ".join(items)
                return f"It is not the case that the following holds: {tail}"

            return f"It is not the case that {R(inner)}"

        if n.op == "AND":
            kids = n.kids or []
            parts = [strip_period(R(k)) for k in kids]
            if len(parts) == 0:
                return ""
            if len(parts) == 1:
                return parts[0]
            sentences = []
            sentences.append(f"First, {parts[0]}.")
            for i, p in enumerate(parts[1:], start=1):
                ord_word = ordinal_words[i] if i < len(ordinal_words) else f"{i+1}th"
                sentences.append(f"and {ord_word}, {p}.")
            return " ".join(sentences).strip()

        if n.op == "OR":
            kids = n.kids or []
            parts = [strip_period(R(k)) for k in kids]
            if len(parts) == 0:
                return ""
            if len(parts) == 1:
                return parts[0]
            items = [f"({ _roman(i+1) }) {p}" for i, p in enumerate(parts)]
            return "either " + " OR ".join(items)

        if n.op == "IMPLIES":
            return f"If {strip_period(R(n.a))}, then {strip_period(R(n.b))}"
        if n.op == "IFF":
            return f"{strip_period(R(n.a))} if and only if {strip_period(R(n.b))}"

        raise ValueError(f"Unknown op: {n.op}")
    return R(node)
def to_nl_en(formula: str, atom_mapper: Optional[Dict[str, str]] = None) -> str:
    ast = Parser(lex(formula)).parse()
    s = render_en_bracketed(ast, atom_mapper=atom_mapper)
    return s if s.endswith(".") else s + "."
    

def load_lexicon_from_wordnet_or_fallback(seed: int = 0):
    rng = random.Random(seed)
    try:
        from nltk.corpus import wordnet as wn
        def pick_lemmas(pos_tag, k=1000):
            out = set()
            for syn in wn.all_synsets(pos=pos_tag):
                for l in syn.lemmas():
                    name = l.name().replace("_", " ").lower()
                    if name.isalpha() and 3 <= len(name) <= 12:
                        out.add(name)
                        if len(out) >= k: break
                if len(out) >= k: break
            return list(out)
        nouns = pick_lemmas('n', 1500)
        adjs  = pick_lemmas('a', 800)
        verbs = pick_lemmas('v', 900)  
    except Exception:
        print("wrong")
        nouns = ["comet", "sensor", "nebula", "module", "engine", "valve", "signal", "beacon", "galaxy", "circuit"]
        adjs  = ["active", "luminous", "stable", "broken", "noisy", "silent", "charged", "cold", "warm", "ready"]
        verbs = ["rotates", "activates", "fails", "appears", "stops", "vibrates", "drifts", "oscillates"]
    return {"nouns": nouns, "adjs": adjs, "verbs": verbs, "rng": rng}

def make_atom_sentence(name: str, L: dict) -> str:
    rng = L["rng"]
    nouns, adjs, verbs = L["nouns"], L["adjs"], L["verbs"]
    tpl = rng.choice(["noun_is_adj", "noun_verb", "noun_occurs"])

    if tpl == "noun_is_adj":
        return f"The {rng.choice(nouns)} is {rng.choice(adjs)}"

    elif tpl == "noun_verb":
        v = rng.choice(verbs)
        v3 = to_3sg(v)  
        return f"The {rng.choice(nouns)} {v3}"

    else:  # noun_occurs
        return f"The {rng.choice(nouns)} occurs"



def process_jsonl_add_nl_nodes(in_path: str, out_path: str):
    total = 0
    updated = 0
    with open(in_path, "r", encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for raw in fin:
            line = raw.strip()
            if not line:
                continue
            total += 1
            try:
                obj = json.loads(line)
            except Exception as e:
                fout.write(json.dumps({"_parse_error": str(e), "_raw": line}, ensure_ascii=False) + "\n")
                continue

            src_nodes = obj.get("source_nodes")
            if not isinstance(src_nodes, list):
                fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
                continue


            atoms_in_line: Set[str] = set()
            for node_str in src_nodes:
                try:
                    ast = Parser(lex(str(node_str))).parse()
                    collect_atoms(ast, atoms_in_line)
                except Exception:
                    pass

            stable_key = json.dumps(sorted(src_nodes), ensure_ascii=False)
            seed_for_line = int(hashlib.md5(stable_key.encode("utf-8")).hexdigest()[:8], 16)
            L = load_lexicon_from_wordnet_or_fallback(seed=seed_for_line)

            atom_mapper = {a: make_atom_sentence(a, L) for a in sorted(atoms_in_line)}


            nl_nodes = []
            for node_str in src_nodes:
                try:
                    nl_nodes.append(to_nl_en(str(node_str), atom_mapper=atom_mapper))
                except Exception as e:
                    nl_nodes.append(f"[nl_error: {e}]")

            obj["nl_nodes"] = nl_nodes
            updated += 1
            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

    return {"total_lines": total, "lines_with_source_nodes": updated}

In [152]:
process_jsonl_add_nl_nodes(
    "rewrite_path3.dedup_10.jsonl",
    "rewrite_path3.dedup_10_wn2.jsonl"
)

{'total_lines': 10, 'lines_with_source_nodes': 10}